In [1]:
from convectio import Mitten

In [2]:
test1 = Mitten(([7, 8]), ([3, 4, 5, 6, 7], [3, 4, 5]), dirty_qc=False)

Configured IOP 7 with Transects: [3, 4, 5, 6, 7]
Configured IOP 8 with Transects: [3, 4, 5]
Success! Loaded 15468 time steps.
Transects available: ['IOP07_T03_e' 'IOP07_T04_w' 'IOP07_T05_e' 'IOP07_T06_w' 'IOP07_T07_e'
 'IOP08_T03_e' 'IOP08_T04_w' 'IOP08_T05_e']


In [3]:
test = test1.extract_tr()
t = test1.transect_dict()

In [4]:
from convectio import rel_distance

ds = rel_distance(test, t)
ds

<xarray.Dataset> Size: 3MB
Dimensions:       (time: 15468, ls_distance: 32, fr_distance: 15468)
Coordinates:
  * time          (time) datetime64[ns] 124kB 2024-07-11T14:59:00 ... 2024-07...
    transect_id   (time) object 124kB 'IOP07_T03_e' ... 'IOP08_T05_e'
  * ls_distance   (ls_distance) float64 256B 1.687 1.656 1.625 ... 2.271 2.427
  * fr_distance   (fr_distance) float64 124kB -3.908 -3.907 ... 41.65 41.65
Data variables: (12/17)
    alt           (time) float32 62kB 202.8 202.7 202.5 ... 229.9 229.9 230.0
    lat           (time) float32 62kB 42.71 42.71 42.71 ... 42.75 42.75 42.75
    lon           (time) float32 62kB -86.2 -86.2 -86.2 ... -85.79 -85.79 -85.79
    fast_temp     (time) float32 62kB 295.6 295.6 295.6 ... 300.1 300.1 300.1
    slow_temp     (time) float32 62kB 295.9 295.9 295.9 ... 301.4 301.4 301.4
    pressure      (time) float32 62kB 9.96e+04 9.961e+04 ... 9.928e+04 9.928e+04
    ...            ...
    dewpoint      (time) float32 62kB 18.58 18.58 18.58 ... 17.87 17.84 17.78
    mixing_ratio  (time) float32 62kB 0.01366 0.01366 ... 0.01307 0.01302
    theta         (time) float32 62kB 295.9 295.9 295.9 ... 300.8 300.7 300.7
    theta_v       (time) float32 62kB 298.4 298.4 298.4 ... 303.1 303.1 303.1
    theta_e       (time) float32 62kB 334.5 334.5 334.5 ... 338.6 338.5 338.4
    error_flag    (time) <U34 2MB 'g00-p02-tf00-ts00-rh00-f01-w08-a00' ... 'g...
Attributes:
    title:        2024-07-11 Combined Mesonet and Tracker synchronized data file
    institution:  Central Michigan University
    source:       Combined Mesonet and Tracker alpha (CoMeT-alpha)
    comment:      PI Contact Info: Jason Keeler (keele1j@cmich.edu)
    description:  All selected transects.

In [4]:
import xarray as xr
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

def prep_binned_boxplot(ds, bin_var, data_var, bin_step):
    """
    Bins an xarray dataset variable by a coordinate/variable for box plotting.

    Parameters:
    -----------
    ds : xarray.Dataset
        The input dataset.
    bin_var : str
        The name of the variable/coordinate to bin by (e.g., 'distance').
    data_var : str
        The name of the data variable to plot (e.g., 'temperature').
    bin_step : int or float
        The size of each bin in the units of bin_var (e.g., every 10 km).

    Returns:
    --------
    pd.DataFrame
        A dataframe containing the raw values and a 'bin' column,
        ready for seaborn or matplotlib plotting.
    """

    # 1. Select only necessary variables and convert to dataframe
    # This flattens the arrays (handling multidimensional data automatically)
    df = ds[[bin_var, data_var]].to_dataframe().dropna()

    # 2. Define the bins
    min_val = np.floor(df[bin_var].min())
    max_val = np.ceil(df[bin_var].max())

    # Create bin edges (e.g., 0, 10, 20, 30...)
    bins = np.arange(min_val, max_val + bin_step, bin_step)

    # 3. Bin the data using pandas cut
    # This creates a new categorical column assigning each row to a bin
    df['bin'] = pd.cut(df[bin_var], bins=bins)

    # Optional: Clean up the dataframe (reset index if it was multi-index)
    df = df.reset_index(drop=True)

    return df

In [6]:
prep_binned_boxplot(ds, bin_var='fr_dist', data_var='theta_e', bin_step=10)

KeyError: 'fr_dist'